# LLM Project: Building a News Research Tool

**Equity Research News Summariser** — powered by LangChain, OpenAI, and NewsAPI.

---

### Project Structure
```
Project9/
├── news_research_tool.ipynb  ← This notebook (exploration + testing)
├── langchain_config.py       ← Core logic module (imported by app)
├── app.py                    ← Streamlit web UI (run separately)
├── requirements.txt          ← Dependencies
└── .env                      ← API keys (create from env.example)
```

> **Note on Streamlit in notebooks:** Streamlit (`st.*`) is a standalone web server and cannot render inside Jupyter cells. All core logic (LangChain, NewsAPI, LLM summarisation) runs fully in this notebook. To see the web UI, run `streamlit run app.py` from a terminal.


---
## Phase 1: Environment Setup

### Task 1.1 — Install Required Libraries

Run this cell once to install all dependencies.

In [ ]:
# Install all project dependencies
#!pip install langchain langchain-openai openai streamlit newsapi-python python-dotenv --quiet
#%pip install langchain langchain-community langchain-core
#%pip install langchain-openai
#from langchain_core.prompts import PromptTemplate, ChatPromptTemplate
#%pip install newsapi-python

### Task 1.2 — Configure API Keys

Create a `.env` file in the project folder (copy from `env.example`) with your keys:
```
OPENAI_API_KEY=sk-...
NEWSAPI_KEY=xxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxx
```

- OpenAI key: https://platform.openai.com/api-keys
- NewsAPI key: https://newsapi.org/register

The cell below loads them via `python-dotenv`. **Never hard-code keys in code.**

In [17]:
import os
from dotenv import load_dotenv
from sympy import python

load_dotenv(dotenv_path="Keys.env")



OPENAI_API_KEY = os.getenv("OPENAI_API_KEY")
NEWSAPI_KEY    = os.getenv("NEWSAPI_KEY")  #why it is not reading the key from .env file?


# Validate keys are loaded (shows masked values, not the actual key)
print("OPENAI_API_KEY:", "[SET]" if OPENAI_API_KEY else "[MISSING — check .env]")
print("NEWSAPI_KEY   :", "[SET]" if NEWSAPI_KEY    else "[MISSING — check .env]")

OPENAI_API_KEY: [SET]
NEWSAPI_KEY   : [SET]


---
## Phase 2: LangChain Configuration

### Task 2.1 & 2.2 — Initialise the OpenAI LLM and PromptTemplate

In [16]:
from langchain_openai import OpenAI
from langchain_core.prompts import PromptTemplate
from langchain_openai import ChatOpenAI

# Initialise the LLM
#llm = OpenAI(api_key=OPENAI_API_KEY, temperature=0.7)
llm = ChatOpenAI(model="gpt-4", temperature=0.7, api_key=OPENAI_API_KEY)
# Prompt template — takes query + news summaries, returns analyst-grade summary
template = """
You are an AI assistant helping an equity research analyst. Given the following query
and the provided news article summaries, provide a concise and insightful overall summary.

Query: {query}

Summaries: {summaries}

Overall Summary:
"""

prompt = PromptTemplate(template=template, input_variables=["query", "summaries"])

# LangChain Expression Language (LCEL) chain: prompt → llm
chain = prompt | llm

print("LLM and chain initialised successfully.")
print(f"LLM model: {llm.model_name}")

LLM and chain initialised successfully.
LLM model: gpt-4


**Quick sanity test** — invoke the chain without any news (just to confirm the LLM connection works):

In [ ]:
# Sanity test: call the LLM directly with a simple prompt
test_response = llm.invoke("Say 'LLM connection successful!' and nothing else.")
print(test_response.strip())

---
## Phase 3: Core Logic (NewsAPI Integration)

The Streamlit interface runs separately (`streamlit run app.py`).  
Here we build and test the same core functions that `langchain_config.py` exposes.

### Task 4.1 — Fetch Articles from NewsAPI

In [6]:
from newsapi import NewsApiClient

newsapi = NewsApiClient(api_key=NEWSAPI_KEY)

def get_news_articles(query: str, page_size: int = 10) -> list:
    """Fetch up to `page_size` English articles sorted by relevancy."""
    response = newsapi.get_everything(
        q=query,
        language="en",
        sort_by="relevancy",
        page_size=page_size
    )
    return response.get("articles", [])


def summarize_articles(articles: list) -> str:
    """Concatenate article descriptions into a single block of text."""
    descriptions = [
        article["description"]
        for article in articles
        if article.get("description")
    ]
    return " ".join(descriptions) if descriptions else "No article descriptions available."


def get_summary(query: str):
    """Return (articles, concatenated_summaries) for a query."""
    articles = get_news_articles(query)
    if not articles:
        return None, "No articles found for this query."
    return articles, summarize_articles(articles)


print("NewsAPI functions defined.")

NewsAPI functions defined.


### Test: Fetch articles for a sample query

In [7]:
SAMPLE_QUERY = "Apple earnings Q3 2024"

articles, summaries = get_summary(SAMPLE_QUERY)

if articles is None:
    print(summaries)   # error message
else:
    print(f"Found {len(articles)} articles for query: '{SAMPLE_QUERY}'\n")
    for i, a in enumerate(articles, 1):
        print(f"{i:2}. [{a.get('publishedAt','')[:10]}] {a.get('title','No title')}")
        print(f"    Source: {a.get('source', {}).get('name', 'Unknown')}")

Found 5 articles for query: 'Apple earnings Q3 2024'

 1. [2026-10-15] Digest: Netflix Cuts 6%of Staff; UK Marketing Budgets Hit a Two-Year High in Q3
    Source: ExchangeWire
 2. [2026-10-15] Global wave of selling as traders flee tech stocks—and futures are signaling the bloodbath will continue today
    Source: Fortune
 3. [2026-10-22] Elon Musk’s dream merger is a nightmare for SpaceX shareholders
    Source: Fortune
 4. [2026-10-13] Rivian’s Mind Robotics snags Doug Field, ex-Tesla and Ford EV chief
    Source: Electrek
 5. [2026-10-15] What’s up with… Smartphone sales, Austrian APIs, Jio
    Source: TelecomTV


---
## Phase 4: End-to-End Pipeline Test

Combines NewsAPI + LangChain → final analyst summary.

In [8]:
def run_pipeline(query: str) -> str:
    """Full pipeline: fetch articles → extract descriptions → LLM summarise."""
    articles, raw_summaries = get_summary(query)
    
    if articles is None:
        return raw_summaries  # error string
    
    # Invoke the LangChain chain
    final_summary = chain.invoke({"query": query, "summaries": raw_summaries})
    return final_summary.strip()

In [ ]:
# Run the full pipeline on the sample query
summary = run_pipeline(SAMPLE_QUERY)

print(f"Query: {SAMPLE_QUERY}")
print("-" * 60)
print(summary)

---
## Phase 5: Testing & Validation

### Task 5.1 — Test Multiple Queries

In [11]:
test_queries = [
    "Federal Reserve interest rate decision 2024",
    "Tesla quarterly earnings revenue",
    "AI chip semiconductor market outlook",
]

results = {}

for q in test_queries:
    print(f"\nProcessing: '{q}'")
    articles, _ = get_summary(q)
    article_count = len(articles) if articles else 0
    results[q] = {"article_count": article_count, "status": "OK" if article_count > 0 else "No results"}
    print(f"  Articles found: {article_count}  |  Status: {results[q]['status']}")


Processing: 'Federal Reserve interest rate decision 2024'
  Articles found: 10  |  Status: OK

Processing: 'Tesla quarterly earnings revenue'
  Articles found: 10  |  Status: OK

Processing: 'AI chip semiconductor market outlook'
  Articles found: 10  |  Status: OK


### Task 5.2 — Validate Summary Quality

Evaluate the summary on one query and check basic quality criteria.

In [ ]:
EVAL_QUERY = "Federal Reserve interest rate decision 2024"

articles, raw = get_summary(EVAL_QUERY)
response     = chain.invoke({"query": EVAL_QUERY, "summaries": raw}).strip()

print("=== SUMMARY ===")
print(response)
print()

# Basic quality checks
checks = {
    "Non-empty response"          : len(response) > 50,
    "Contains query keyword"      : any(kw in response.lower() for kw in ["federal", "rate", "interest", "fed"]),
    "Does not echo the prompt"    : "Overall Summary:" not in response,
    "Reasonable length (< 2000c)" : len(response) < 2000,
}

print("=== QUALITY CHECKS ===")
all_passed = True
for check, passed in checks.items():
    icon = "PASS" if passed else "FAIL"
    print(f"  [{icon}] {check}")
    if not passed:
        all_passed = False

print()
print("Overall result:", "ALL CHECKS PASSED" if all_passed else "SOME CHECKS FAILED")

---
## Interactive Demo (Notebook UI)

A lightweight in-notebook interface using `ipywidgets` — equivalent to the Streamlit app but runs inside Jupyter.

In [ ]:
try:
    import ipywidgets as widgets
    from IPython.display import display, Markdown, clear_output

    query_box = widgets.Text(
        placeholder="e.g. Apple earnings Q3 2024",
        description="Query:",
        layout=widgets.Layout(width="60%")
    )
    run_btn    = widgets.Button(description="Get News Summary", button_style="primary")
    out        = widgets.Output()

    def on_click(b):
        with out:
            clear_output(wait=True)
            q = query_box.value.strip()
            if not q:
                print("Please enter a query.")
                return
            print(f"Fetching articles for: '{q}' ...")
            arts, raw = get_summary(q)
            if arts is None:
                print(raw)
                return
            print(f"Found {len(arts)} articles. Generating summary...")
            summary = chain.invoke({"query": q, "summaries": raw}).strip()
            display(Markdown(f"### Summary\n{summary}"))
            display(Markdown("### Source Articles"))
            for i, a in enumerate(arts, 1):
                title = a.get('title', 'No title')
                source = a.get('source', {}).get('name', 'Unknown')
                date   = a.get('publishedAt', '')[:10]
                desc   = a.get('description', 'No description.')
                url    = a.get('url', '')
                link   = f" | [Read]({url})" if url else ""
                display(Markdown(f"**{i}. {title}**  \n*{source} — {date}*{link}  \n{desc}"))

    run_btn.on_click(on_click)
    display(widgets.VBox([query_box, run_btn, out]))

except ImportError:
    print("ipywidgets not available. Run: pip install ipywidgets")
    print("Alternatively use run_pipeline(query) directly:")
    print("  summary = run_pipeline('Apple earnings Q3 2024')")
    print("  print(summary)")

---
## Phase 6: Running the Full Streamlit App

The notebook covers all the core logic. To launch the polished web UI:

1. Create your `.env` file from `env.example` with real API keys.
2. Open a terminal in the project folder and run:

```bash
streamlit run app.py 
python -m streamlit run app.py
```

3. A browser window will open at `http://localhost:8501`.

> Streamlit cannot run inside a Jupyter notebook — it is a standalone web server that requires a terminal process.


---
## Optional Enhancements (Task 7)

### Task 7.3 — Export Summaries to CSV

In [14]:
import pandas as pd
from datetime import datetime

def export_results(query: str, output_path: str = None):
    """Fetch articles for `query`, generate LLM summary, and save to CSV."""
    articles, raw = get_summary(query)
    if articles is None:
        print(raw)
        return

    llm_summary = chain.invoke({"query": query, "summaries": raw}).strip()

    rows = []
    for a in articles:
        rows.append({
            "query"       : query,
            "title"       : a.get("title"),
            "source"      : a.get("source", {}).get("name"),
            "published_at": a.get("publishedAt", "")[:10],
            "description" : a.get("description"),
            "url"         : a.get("url"),
            "llm_summary" : llm_summary,
        })

    df = pd.DataFrame(rows)
    if output_path is None:
        ts = datetime.now().strftime("%Y%m%d_%H%M%S")
        output_path = f"news_summary_{ts}.csv"

    df.to_csv(output_path, index=False)
    print(f"Exported {len(df)} rows to {output_path}")
    return df


# Uncomment to run:
# df = export_results("Apple earnings Q3 2024")
# df.head()

---
## Project Summary

| Component | File | Notebook cell |
|---|---|---|
| LLM initialisation | `langchain_config.py` | Phase 2 |
| NewsAPI integration | `langchain_config.py` | Phase 3 |
| Full pipeline | `langchain_config.py` | Phase 4 |
| Testing & validation | — | Phase 5 |
| Interactive demo | — | Interactive Demo |
| Web UI | `app.py` | Run separately |
| Export to CSV | — | Optional Enhancements |

**Dependencies:** `langchain`, `langchain-openai`, `openai`, `streamlit`, `newsapi-python`, `python-dotenv`
